In [1]:
! pip install -q torch-adopt optuna datasets==3.6.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.6/386.6 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.5/242.5 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Param Tuning

### Hyperparameter Optimization

In [ ]:
from typing import Any, Callable, Dict, Literal, Tuple

import optuna
import torch
import torch.optim as optim
import torch.utils.data as data
from adopt import ADOPT
from torch import nn


OptimizerName = Literal["Adam", "AdamW", "ADOPT", "RMSprop", "SGD"]
ModelFactory = Callable[..., nn.Module]
GANModelFactory = Callable[..., Tuple[nn.Module, nn.Module]]
DataLoaderFactory = Callable[..., Tuple[data.DataLoader, data.DataLoader, data.DataLoader]]
TrainEvalLoop = Callable[..., float]
GANTrainEvalLoop = Callable[..., float]


class ParamOptimizer:
    def __init__(self):
        self.results = {}

    def _get_optimizer_search_space(
        self,
        optimizer_name: OptimizerName,
        trial: optuna.Trial,
        prefix: str = "",
    ) -> Dict[str, Any]:

        # function to allow for optimizing GANs
        def param_name(name: str) -> str:
            return f"{prefix}{name}" if prefix else name

        if optimizer_name == 'ADOPT':
            return {
                'lr': trial.suggest_float(param_name('lr'), 1e-5, 1e-2, log=True),
                'betas': (
                    trial.suggest_float(param_name('beta1'), 0.8, 0.99),
                    trial.suggest_float(param_name('beta2'), 0.9, 0.9999)
                ),
                'weight_decay': trial.suggest_float(param_name('weight_decay'), 0.0, 0.1, log=False),
                'decouple': True,
            }

        elif optimizer_name == 'Adam':
            return {
                'lr': trial.suggest_float(param_name('lr'), 1e-5, 1e-2, log=True),
                'betas': (
                    trial.suggest_float(param_name('beta1'), 0.8, 0.99),
                    trial.suggest_float(param_name('beta2'), 0.9, 0.999)
                ),
                'weight_decay': trial.suggest_float(param_name('weight_decay'), 0.0, 1e-2, log=False),
            }

        elif optimizer_name == 'AdamW':
            return {
                'lr': trial.suggest_float(param_name('lr'), 1e-5, 1e-2, log=True),
                'betas': (
                    trial.suggest_float(param_name('beta1'), 0.8, 0.99),
                    trial.suggest_float(param_name('beta2'), 0.9, 0.999)
                ),
                'weight_decay': trial.suggest_float(param_name('weight_decay'), 0.0, 0.1, log=False),
            }

        elif optimizer_name == 'RMSprop':
            return {
                'lr': trial.suggest_float(param_name('lr'), 1e-5, 1e-2, log=True),
                'alpha': trial.suggest_float(param_name('alpha'), 0.9, 0.999),
                'weight_decay': trial.suggest_float(param_name('weight_decay'), 0.0, 1e-2, log=False),
                'momentum': trial.suggest_float(param_name('momentum'), 0.0, 0.1),
            }

        elif optimizer_name == 'SGD':
            return {
                'lr': trial.suggest_float(param_name('lr'), 1e-5, 1e-2, log=True),
                'momentum': trial.suggest_float(param_name('momentum'), 0.8, 0.99),
                'weight_decay': trial.suggest_float(param_name('weight_decay'), 0.0, 1e-2, log=False),
            }

        else:
            raise ValueError(f"Unknown optimizer: {optimizer_name}")

    def _create_optimizer(
        self,
        optimizer_name: str,
        model: nn.Module,
        **kwargs,
    ) -> optim.Optimizer:

        model_params = model.parameters()

        if optimizer_name == 'SGD':
            return optim.SGD(model_params, **kwargs)
        elif optimizer_name == 'RMSprop':
            return optim.RMSprop(model_params, **kwargs)
        elif optimizer_name == 'Adam':
            return optim.Adam(model_params, **kwargs)
        elif optimizer_name == 'AdamW':
            return optim.AdamW(model_params, **kwargs)
        elif optimizer_name in ['ADOPT']:
            return ADOPT(model_params, **kwargs)
        else:
            raise ValueError(f"Unknown optimizer: {optimizer_name}")

    def _objective_factory(
        self,
        optimizer_name: OptimizerName,
        model_factory: ModelFactory,
        dataloader_factory: DataLoaderFactory,
        train_eval_loop: TrainEvalLoop,
        device: str = "cuda",
    ) -> Callable[[optuna.Trial], float]:

        def objective(trial: optuna.Trial) -> float:
            optimizer_params = self._get_optimizer_search_space(optimizer_name, trial)
            model = model_factory().to(device)
            optimizer = self._create_optimizer(optimizer_name, model, **optimizer_params)
            train_loader, val_loader, _ = dataloader_factory()

            val_metric = train_eval_loop(
                model=model,
                optimizer=optimizer,
                train_loader=train_loader,
                val_loader=val_loader,
                trial=trial,
                device=device,
            )

            return val_metric

        return objective

    def _gan_objective_factory(
        self,
        optimizer_name: OptimizerName,
        gan_model_factory: GANModelFactory,
        dataloader_factory: DataLoaderFactory,
        gan_train_eval_loop: GANTrainEvalLoop,
        device: str = "cuda",
    ) -> Callable[[optuna.Trial], float]:

        def objective(trial: optuna.Trial) -> float:
            gen_params = self._get_optimizer_search_space(optimizer_name, trial, prefix="gen_")
            disc_params = self._get_optimizer_search_space(optimizer_name, trial, prefix="disc_")

            generator, discriminator = gan_model_factory()
            generator = generator.to(device)
            discriminator = discriminator.to(device)

            gen_optimizer = self._create_optimizer(optimizer_name, generator, **gen_params)
            disc_optimizer = self._create_optimizer(optimizer_name, discriminator, **disc_params)

            train_loader, val_loader, _ = dataloader_factory()

            val_metric = gan_train_eval_loop(
                generator=generator,
                discriminator=discriminator,
                gen_optimizer=gen_optimizer,
                disc_optimizer=disc_optimizer,
                train_loader=train_loader,
                val_loader=val_loader,
                trial=trial,
                device=device,
            )

            return val_metric

        return objective

    def optimize_optimizer_params(
        self,
        optimizer_name: OptimizerName,
        model_factory: ModelFactory,
        dataloader_factory: DataLoaderFactory,
        train_eval_loop: TrainEvalLoop,
        *,
        n_trials: int = 20,
        n_startup_trials: int = 5,
        n_warmup_steps: int = 3,
        interval_steps: int = 2,
        timeout: int | None = None,
        device: str = "cuda",
    ) -> optuna.Study:
        """Optimize hyperparameters for a specific optimizer on a single task."""

        print(f"\nOptimizing {optimizer_name}...")

        study = optuna.create_study(
            direction='maximize',
            sampler=optuna.samplers.TPESampler(seed=42),
            pruner=optuna.pruners.MedianPruner(
                n_startup_trials=n_startup_trials,
                n_warmup_steps=n_warmup_steps,
                interval_steps=interval_steps,
            )
        )

        objective = self._objective_factory(
            optimizer_name,
            model_factory,
            dataloader_factory,
            train_eval_loop,
            device,
        )
        study.optimize(objective, n_trials=n_trials, timeout=timeout)

        self.results[optimizer_name] = {
            'best_params': study.best_params,
            'best_value': study.best_value,
            'n_trials': len(study.trials),
            'study': study,
        }

        print(f"{optimizer_name} - Best value: {study.best_value:.4f}")
        print(f"{optimizer_name} - Best params: {study.best_params}")

        return study

    def optimize_gan_optimizers_params(
        self,
        optimizer_name: OptimizerName,
        gan_model_factory: GANModelFactory,
        dataloader_factory: DataLoaderFactory,
        gan_train_eval_loop: GANTrainEvalLoop,
        *,
        n_trials: int = 20,
        n_startup_trials: int = 5,
        n_warmup_steps: int = 3,
        interval_steps: int = 2,
        timeout: int | None = None,
        device: str = "cuda",
    ) -> optuna.Study:
        """Optimize hyperparameters for GAN training with separate optimizers for generator and discriminator."""

        print(f"\nOptimizing GAN with {optimizer_name}...")

        study = optuna.create_study(
            direction='maximize',
            sampler=optuna.samplers.TPESampler(seed=42),
            pruner=optuna.pruners.MedianPruner(
                n_startup_trials=n_startup_trials,
                n_warmup_steps=n_warmup_steps,
                interval_steps=interval_steps,
            )
        )

        objective = self._gan_objective_factory(
            optimizer_name,
            gan_model_factory,
            dataloader_factory,
            gan_train_eval_loop,
            device,
        )
        study.optimize(objective, n_trials=n_trials, timeout=timeout)

        gan_key = f"GAN_{optimizer_name}"
        self.results[gan_key] = {
            'best_params': study.best_params,
            'best_value': study.best_value,
            'n_trials': len(study.trials),
            'study': study,
        }

        print(f"GAN {optimizer_name} - Best value: {study.best_value:.4f}")
        print(f"GAN {optimizer_name} - Best params: {study.best_params}")

        return study

    def get_best_params_dict(
        self,
        optimizer_name: OptimizerName | None = None
    ) -> Dict[str, Any]:
        """Get the best parameters for an optimizer."""

        if optimizer_name:
            return self.results[optimizer_name]
        else:
            return self.results

### Image Classification on CIFAR100

In [ ]:
import torchvision
from torchvision import models, transforms, datasets

CIFAR100_ROOT = "/data/cifar100"

def cifar100_model_factory(num_classes: int = 100) -> nn.Module:
    model = models.resnet18(num_classes=num_classes)
    # modify first conv layer to avoid upscaling to 224x224
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    return model

def cifar100_dataloader_factory(
    batch_size: int = 32,
    seed: int = 42
) -> Tuple[data.DataLoader, data.DataLoader, data.DataLoader]:
    # standard CIFAR-100 mean and std
    mean = [0.5071, 0.4865, 0.4409]
    std = [0.2673, 0.2564, 0.2761]

    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])

    temp_dataset = datasets.CIFAR100(root=CIFAR100_ROOT, train=True, transform=transform, download=True)
    test_dataset = datasets.CIFAR100(root=CIFAR100_ROOT, train=False, transform=transform, download=True)

    train_size = int(0.8 * len(temp_dataset))
    val_size = len(temp_dataset) - train_size
    generator = torch.Generator().manual_seed(seed)
    train_dataset, val_dataset = data.random_split(temp_dataset, [train_size, val_size], generator=generator)

    train_loader = data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader

In [ ]:
from sklearn.metrics import f1_score

def cifar100_train_eval_loop(
    *,
    model: nn.Module,
    optimizer: optim.Optimizer,
    train_loader: data.DataLoader,
    val_loader: data.DataLoader,
    trial: optuna.Trial | None = None,
    criterion: nn.Module = nn.CrossEntropyLoss(),
    epochs: int = 10,
    device: str = "cuda",
) -> float:
    model = model.to(device)

    for epoch in range(epochs):
        model.train()
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)

            loss.backward()
            optimizer.step()

        model.eval()
        all_preds = []
        all_targets = []
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                preds = torch.argmax(outputs, dim=1)
                all_preds.extend(preds.cpu().tolist())
                all_targets.extend(targets.cpu().tolist())

        f1 = f1_score(all_targets, all_preds, average="macro")
        print(f"Epoch {epoch+1}/{epochs} — F1 Score: {f1:.4f}")

        if trial:
            trial.report(f1, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

    return f1

#### Test factory functions

In [ ]:
model = cifar100_model_factory()
optimizer = optim.Adam(model.parameters())

In [ ]:
train_loader, val_loader, test_loader = cifar100_dataloader_factory()

In [ ]:
final_f1 = cifar100_train_eval_loop(
    model=model,
    optimizer=optimizer,
    train_loader=train_loader,
    val_loader=val_loader,
    device="cuda",
)

Epoch 1/10 — F1 Score: 0.1643
Epoch 2/10 — F1 Score: 0.3289
Epoch 3/10 — F1 Score: 0.3885
Epoch 4/10 — F1 Score: 0.4651
Epoch 5/10 — F1 Score: 0.4748
Epoch 6/10 — F1 Score: 0.5028
Epoch 7/10 — F1 Score: 0.5075
Epoch 8/10 — F1 Score: 0.5161
Epoch 9/10 — F1 Score: 0.5093
Epoch 10/10 — F1 Score: 0.5046


#### Find best parames

In [ ]:
param_optimizer = ParamOptimizer()

In [ ]:
adam_study = param_optimizer.optimize_optimizer_params(
    "Adam",
    cifar100_model_factory,
    cifar100_dataloader_factory,
    cifar100_train_eval_loop,
    n_trials=10,
    n_startup_trials=3
)

[I 2025-05-25 19:55:13,650] A new study created in memory with name: no-name-0de0c8f6-4aa5-4502-96ee-ab0ca84d5cee



Optimizing Adam...
Epoch 1/10 — F1 Score: 0.2154
Epoch 2/10 — F1 Score: 0.2944
Epoch 3/10 — F1 Score: 0.3492
Epoch 4/10 — F1 Score: 0.3938
Epoch 5/10 — F1 Score: 0.4330
Epoch 6/10 — F1 Score: 0.4401
Epoch 7/10 — F1 Score: 0.4480
Epoch 8/10 — F1 Score: 0.4815
Epoch 9/10 — F1 Score: 0.4726


[I 2025-05-25 20:03:19,677] Trial 0 finished with value: 0.48070755828698 and parameters: {'lr': 0.0001329291894316216, 'beta1': 0.9806357182178841, 'beta2': 0.9724674002393291, 'weight_decay': 0.005986584841970366}. Best is trial 0 with value: 0.48070755828698.


Epoch 10/10 — F1 Score: 0.4807
Epoch 1/10 — F1 Score: 0.1443
Epoch 2/10 — F1 Score: 0.2201
Epoch 3/10 — F1 Score: 0.2719
Epoch 4/10 — F1 Score: 0.3184
Epoch 5/10 — F1 Score: 0.3355
Epoch 6/10 — F1 Score: 0.3411
Epoch 7/10 — F1 Score: 0.3471
Epoch 8/10 — F1 Score: 0.3501
Epoch 9/10 — F1 Score: 0.3484


[I 2025-05-25 20:11:23,598] Trial 1 finished with value: 0.3397166263187554 and parameters: {'lr': 2.9380279387035334e-05, 'beta1': 0.8296389588638785, 'beta2': 0.9057502776046518, 'weight_decay': 0.008661761457749353}. Best is trial 0 with value: 0.48070755828698.


...

Epoch 4/10 — F1 Score: 0.3469
Epoch 1/10 — F1 Score: 0.1044
Epoch 2/10 — F1 Score: 0.1667
Epoch 3/10 — F1 Score: 0.2099


[I 2025-05-25 20:48:28,249] Trial 8 pruned. 


Epoch 4/10 — F1 Score: 0.2494
Epoch 1/10 — F1 Score: 0.2075
Epoch 2/10 — F1 Score: 0.3112
Epoch 3/10 — F1 Score: 0.3557


[I 2025-05-25 20:51:41,464] Trial 9 pruned. 


Epoch 4/10 — F1 Score: 0.3659
Adam - Best value: 0.4852
Adam - Best params: {'lr': 0.00023345864076016249, 'beta1': 0.9491834326646726, 'beta2': 0.9197677044336776, 'weight_decay': 0.005142344384136116}


In [ ]:
param_optimizer.get_best_params_dict("Adam")

{'best_params': {'lr': 0.00023345864076016249,
  'beta1': 0.9491834326646726,
  'beta2': 0.9197677044336776,
  'weight_decay': 0.005142344384136116},
 'best_value': 0.4851962085270538,
 'n_trials': 10,
 'study': <optuna.study.study.Study at 0x7c3c6382a610>}

### Sentiment Classification on IMDb

In [5]:
from typing import Tuple

import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils import data
from transformers import (
    DistilBertForSequenceClassification,
    DistilBertTokenizer,
    default_data_collator,
)


TOKENIZER_NAME = "distilbert-base-uncased"

def imdb_model_factory(num_labels: int) -> nn.Module:
    return DistilBertForSequenceClassification.from_pretrained(
        TOKENIZER_NAME, num_labels=num_labels
    )

def imdb_dataloader_factory(
    dataset_path: str,
    max_dataset_size: int | None = None,
    seed: int = 42,
) -> Tuple[data.DataLoader, data.DataLoader, data.DataLoader]:
    df = pd.read_csv(dataset_path, encoding='latin-1', on_bad_lines='skip')

    df['sentiment'] = df['sentiment'].astype('category')
    print(df['sentiment'].value_counts())
    df_size = len(df)
    if max_dataset_size:
        size = min(max_dataset_size, df_size)
        train_size = int(0.7 * size)
        test_size = int(0.15 * size)
        val_size = size - train_size - test_size
    else:
        train_size = int(0.7 * df_size)
        test_size = int(0.15 * df_size)
        val_size = df_size - train_size - test_size

    df_train, df_temp = train_test_split(
        df, train_size=train_size, test_size=test_size + val_size,
        stratify=df['sentiment'], random_state=seed
    )
    df_test, df_val = train_test_split(
        df_temp, train_size=test_size, test_size=val_size,
        stratify=df_temp['sentiment'], random_state=seed
    )

    df_train.reset_index(drop=True, inplace=True)
    df_val.reset_index(drop=True, inplace=True)
    df_test.reset_index(drop=True, inplace=True)

    label_map = {'negative': 0, 'positive': 1}
    for df_split in [df_train, df_val, df_test]:
        df_split.rename(columns={"sentiment": "label"}, inplace=True)
        df_split["label"] = df_split["label"].map(label_map).astype(int)

    tokenizer = DistilBertTokenizer.from_pretrained(TOKENIZER_NAME)

    def preprocess_function(examples):
        return tokenizer(
            examples["review"],
            truncation=True,
            padding="max_length",
            max_length=512
        )

    train_ds = Dataset.from_pandas(df_train[["review", "label"]])
    val_ds = Dataset.from_pandas(df_val[["review", "label"]])
    test_ds = Dataset.from_pandas(df_test[["review", "label"]])

    train_ds = train_ds.map(preprocess_function, batched=True)
    val_ds = val_ds.map(preprocess_function, batched=True)
    test_ds = test_ds.map(preprocess_function, batched=True)

    train_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])
    val_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])
    test_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])

    train_loader = data.DataLoader(train_ds, batch_size=32, shuffle=True, collate_fn=default_data_collator)
    val_loader = data.DataLoader(val_ds, batch_size=32, shuffle=False, collate_fn=default_data_collator)
    test_loader = data.DataLoader(test_ds, batch_size=32, shuffle=False, collate_fn=default_data_collator)

    return train_loader, val_loader, test_loader

In [16]:
from datetime import datetime

import optuna
import torch
from torch import optim
from sklearn.metrics import f1_score

def imdb_train_eval_loop(
    *,
    model: nn.Module,
    optimizer: optim.Optimizer,
    train_loader: data.DataLoader,
    val_loader: data.DataLoader,
    trial: optuna.Trial | None = None,
    criterion: nn.Module = nn.CrossEntropyLoss(),
    epochs: int = 10,
    device: str = "cuda",
) -> float:
    model = model.to(device)

    for epoch in range(epochs):
        model.train()
        total_train_loss = 0
        for batch in train_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs.logits, labels)
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()

        avg_train_loss = total_train_loss / len(train_loader)

        model.eval()
        all_preds = []
        all_labels = []
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)

                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                preds = torch.argmax(outputs.logits, dim=1)

                all_preds.extend(preds.cpu().tolist())
                all_labels.extend(labels.cpu().tolist())

        f1 = f1_score(all_labels, all_preds, average="macro")
        print(f"Epoch {epoch+1}/{epochs} — F1 Score: {f1:.4f} | Train Loss: {avg_train_loss:.4f}")

        if trial:
            trial.report(f1, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

    return f1


def imdb_test(
    *,
    model: nn.Module,
    test_loader: data.DataLoader,
    criterion: nn.Module = nn.CrossEntropyLoss(),
    device: str = "cuda",
):
    model.to(device)
    model.eval()

    test_start = datetime.now()
    test_loss = 0
    test_preds = []
    test_labels = []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs.logits, labels)
            test_loss += loss.item()
            preds = torch.argmax(outputs.logits, dim=1)

            test_preds.extend(preds.cpu().tolist())
            test_labels.extend(labels.cpu().tolist())

    test_end = datetime.now()
    test_time = (test_end - test_start).total_seconds()
    test_loss /= len(test_loader)
    test_f1_macro = f1_score(test_labels, test_preds, average="macro")

    print("=== TEST RESULTS ===")
    print(f"Test Loss: {test_loss:.4f}")
    print(f"Test F1 Macro: {test_f1_macro:.4f}")
    print(f"Test Time: {test_time:.2f} seconds")

#### Test factory functions

In [9]:
model = imdb_model_factory(num_labels=2)
optimizer = optim.Adam(model.parameters(), lr=2e-5)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
train_loader, val_loader, test_loader = imdb_dataloader_factory("./IMDB_Dataset.csv", max_dataset_size=3000)

sentiment
negative    25000
positive    25000
Name: count, dtype: int64


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/2100 [00:00<?, ? examples/s]

Map:   0%|          | 0/450 [00:00<?, ? examples/s]

Map:   0%|          | 0/450 [00:00<?, ? examples/s]

In [17]:
imdb_test(
    model=model,
    test_loader=test_loader,
    device="cuda"
)

=== TEST RESULTS ===
Test Loss: 0.6939
Test F1 Macro: 0.3333
Test Time: 7.13 seconds


In [18]:
final_f1 = imdb_train_eval_loop(
    model=model,
    optimizer=optimizer,
    train_loader=train_loader,
    val_loader=val_loader,
    device="cuda",
    epochs=5,
)

Epoch 1/5 — F1 Score: 0.8841 | Train Loss: 0.4816
Epoch 2/5 — F1 Score: 0.8908 | Train Loss: 0.2399
Epoch 3/5 — F1 Score: 0.8955 | Train Loss: 0.1082
Epoch 4/5 — F1 Score: 0.8955 | Train Loss: 0.0522
Epoch 5/5 — F1 Score: 0.8932 | Train Loss: 0.0214


In [ ]:
imdb_test(
    model=model,
    test_loader=test_loader,
    device="cuda"
)

### Regression on Real Estate Dataset

In [ ]:
import re
from datetime import datetime
from typing import Tuple

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils import data
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split


class RealEstateNN(nn.Module):
    def __init__(self, input_size):
        super(RealEstateNN, self).__init__()

        self.network = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.network(x)

def realestate_model_factory(input_size: int) -> nn.Module:
    return RealEstateNN(input_size)

In [ ]:
def realestate_preprocess_data(df: pd.DataFrame) -> pd.DataFrame:
    data = df.copy()
    data = data.dropna(subset=['Sale Amount'])

    selected_features = [
        'Assessed Value',
        'Property Type',
        'Residential Type',
        'Town',
        'Date Recorded'
    ]


    data['Latitude'] = data['Location'].str.extract(r'POINT \(([+-]?\d*\.?\d+)\s+([+-]?\d*\.?\d+)\)')[1].astype(float)
    data['Longitude'] = data['Location'].str.extract(r'POINT \(([+-]?\d*\.?\d+)\s+([+-]?\d*\.?\d+)\)')[0].astype(float)
    selected_features.extend(['Latitude', 'Longitude'])

    features_df = data[selected_features + ['Sale Amount']].copy()

    features_df['Date Recorded'] = pd.to_datetime(features_df['Date Recorded'], errors='coerce')
    features_df['Year'] = features_df['Date Recorded'].dt.year
    features_df['Month'] = features_df['Date Recorded'].dt.month
    features_df['Day_of_Year'] = features_df['Date Recorded'].dt.dayofyear
    features_df = features_df.drop('Date Recorded', axis=1)

    numerical_cols = ['Assessed Value', 'Latitude', 'Longitude', 'Year', 'Month', 'Day_of_Year']
    for col in numerical_cols:
        if col in features_df.columns:
            features_df[col] = features_df[col].fillna(features_df[col].median())

    categorical_cols = ['Property Type', 'Residential Type', 'Town']
    for col in categorical_cols:
        if col in features_df.columns:
            features_df[col] = features_df[col].fillna('Unknown')

    label_encoders = {}
    for col in categorical_cols:
        if col in features_df.columns:
            le = LabelEncoder()
            features_df[col] = le.fit_transform(features_df[col].astype(str))
            label_encoders[col] = le

    X = features_df.drop('Sale Amount', axis=1)
    y = features_df['Sale Amount']

    mask = ~(X.isnull().any(axis=1) | y.isnull())
    X = X[mask]
    y = y[mask]

    return X, y, label_encoders

def realestate_dataloader_factory(
    dataset_path: str,
    batch_size: int = 64,
    test_size: float = 0.2,
    val_size: float = 0.2
) -> Tuple[data.DataLoader, data.DataLoader, data.DataLoader]:
    df = pd.read_csv(dataset_path)

    X, y, label_encoders = realestate_preprocess_data(df)

    X_np = X.values.astype(np.float32)
    y_np = y.values.astype(np.float32)

    X_temp, X_test, y_temp, y_test = train_test_split(
        X_np, y_np, test_size=test_size, random_state=42, stratify=None
    )

    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=val_size, random_state=42, stratify=None
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    train_dataset = data.TensorDataset(
        torch.FloatTensor(X_train_scaled),
        torch.FloatTensor(y_train)
    )
    val_dataset = data.TensorDataset(
        torch.FloatTensor(X_val_scaled),
        torch.FloatTensor(y_val)
    )
    test_dataset = data.TensorDataset(
        torch.FloatTensor(X_test_scaled),
        torch.FloatTensor(y_test)
    )

    train_loader = data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    input_size = X_train_scaled.shape[1]
    print(f"Data input size: {input_size}")

    return train_loader, val_loader, test_loader

In [ ]:
from sklearn.metrics import r2_score


def realestate_train_eval_loop(
    *,
    model: nn.Module,
    optimizer: optim.Optimizer,
    train_loader: data.DataLoader,
    val_loader: data.DataLoader,
    trial: optuna.Trial | None = None,
    criterion: nn.Module = nn.MSELoss(),
    epochs: int = 20,
    device: str = "cuda",
) -> float:
    model.to(device)

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0

        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            targets = targets.view(-1, 1)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            train_loss += loss.item()

            loss.backward()
            optimizer.step()

        train_loss /= len(train_loader)

        model.eval()
        val_loss = 0.0
        val_batches = 0
        val_preds = []
        val_targets = []

        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                targets = targets.view(-1, 1)
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                val_loss += loss.item()

                val_preds.extend(outputs.cpu().numpy().flatten())
                val_targets.extend(targets.cpu().numpy().flatten())

        val_loss /= len(val_loader)
        val_r2 = r2_score(val_targets, val_preds)

        print(f"Epoch {epoch+1}/{epochs} — Val R2: {val_r2:.4f} | Val Loss: {val_loss:.4f} | Train Loss: {train_loss:.4f}")

        if trial:
            trial.report(val_r2, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

    return val_r2

In [ ]:
train_loader, val_loader, test_loader = realestate_dataloader_factory(
    "Real_Estate_Sales_2001-2022_GL.csv", batch_size=64
)

<ipython-input-6-50f08728a176>:58: DtypeWarning: Columns (8,9,10,11,12) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(dataset_path)


Data input size: 9


In [ ]:
model = realestate_model_factory(input_size=9)
optimizer = optim.Adam(model.parameters())

In [ ]:
final_val_loss = realestate_train_eval_loop(
    model=model,
    optimizer=optimizer,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=30,
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
)

Epoch 1/30 — Val R2: 0.0048 | Val Loss: 3062282719719.9390 | Train Loss: 39480937315560.1875
Epoch 2/30 — Val R2: 0.0095 | Val Loss: 3047930048428.4385 | Train Loss: 39446310575153.1641
Epoch 3/30 — Val R2: 0.0221 | Val Loss: 3008981023717.5142 | Train Loss: 39413525123273.6875
Epoch 4/30 — Val R2: 0.0073 | Val Loss: 3054443643162.0195 | Train Loss: 39380588420190.0312
Epoch 5/30 — Val R2: 0.0607 | Val Loss: 2890161574184.9414 | Train Loss: 39355023608265.3828
Epoch 6/30 — Val R2: 0.0385 | Val Loss: 2958420960190.7178 | Train Loss: 39319912173711.7500
Epoch 7/30 — Val R2: 0.0703 | Val Loss: 2860657502250.5269 | Train Loss: 39305693793152.9922
Epoch 8/30 — Val R2: 0.1276 | Val Loss: 2684247407384.3408 | Train Loss: 39269648074109.2344
Epoch 9/30 — Val R2: 0.1492 | Val Loss: 2617901409763.8354 | Train Loss: 39245920360626.8750
Epoch 10/30 — Val R2: 0.1238 | Val Loss: 2696124304855.8979 | Train Loss: 39219766862551.4453
Epoch 11/30 — Val R2: 0.1434 | Val Loss: 2635772515628.6719 | Train L

### Image Generation on MedMNIST

In [ ]:
! pip install -q medmnist

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 6.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
from typing import Tuple

import medmnist
import torchvision
from medmnist import INFO, Evaluator
from torchvision import transforms


def medmnist_gan_model_factory() -> Tuple[nn.Module, nn.Module]:
    class MedMNISTGenerator(nn.Module):
        def __init__(
            self,
            latent_dim: int = 100,
            output_channels: int = 3,
            output_size: int = 64
        ):
            super().__init__()
            assert output_size % 16 == 0, "Output size must be divisible by 16"

            self.latent_dim = latent_dim
            self.output_size = output_size
            self.output_channels = output_channels
            self.init_size = output_size // 16  # Initial feature map size (e.g., 4x4 if 64x64 output)

            self.fc = nn.Sequential(
                nn.Linear(latent_dim, 256 * self.init_size * self.init_size),
                nn.ReLU(True)
            )

            self.upsample = nn.Sequential(
                nn.Upsample(scale_factor=2),  # 4 -> 8
                nn.Conv2d(256, 128, kernel_size=3, stride=1, padding=1),
                nn.BatchNorm2d(128),
                nn.ReLU(True),

                nn.Upsample(scale_factor=2),  # 8 -> 16
                nn.Conv2d(128, 64, kernel_size=3, stride=1, padding=1),
                nn.BatchNorm2d(64),
                nn.ReLU(True),

                nn.Upsample(scale_factor=2),  # 16 -> 32
                nn.Conv2d(64, 32, kernel_size=3, stride=1, padding=1),
                nn.BatchNorm2d(32),
                nn.ReLU(True),

                nn.Upsample(scale_factor=2),  # 32 -> 64
                nn.Conv2d(32, output_channels, kernel_size=3, stride=1, padding=1),
                nn.Sigmoid()
            )

        def forward(self, z: torch.Tensor) -> torch.Tensor:
            out = self.fc(z)
            out = out.view(z.size(0), 256, self.init_size, self.init_size)
            img = self.upsample(out)
            return img

    class MedMNISTDiscriminator(nn.Module):
        def __init__(
            self,
            input_channels: int = 3,
            input_size: int = 64
        ):
            super().__init__()
            assert input_size % 16 == 0, "Input size must be divisible by 16"

            self.input_channels = input_channels
            self.input_size = input_size
            self.final_size = input_size // 16  # 4 downsampling steps 64 → 32 → 16 → 8 → 4

            self.feature_extractor = nn.Sequential(
                nn.Conv2d(input_channels, 64, kernel_size=4, stride=2, padding=1),  # 64x64 → 32x32
                nn.LeakyReLU(0.2, inplace=True),

                nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),  # 32x32 → 16x16
                nn.BatchNorm2d(128),
                nn.LeakyReLU(0.2, inplace=True),

                nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),  # 16x16 → 8x8
                nn.BatchNorm2d(256),
                nn.LeakyReLU(0.2, inplace=True),

                nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1),  # 8x8 → 4x4
                nn.BatchNorm2d(512),
                nn.LeakyReLU(0.2, inplace=True),
            )

            self.classifier = nn.Sequential(
                nn.Flatten(),
                nn.Linear(512 * self.final_size * self.final_size, 1),
                nn.Sigmoid(),
            )

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            features = self.feature_extractor(x)
            out = self.classifier(features)
            return out

    generator = MedMNISTGenerator()
    discriminator = MedMNISTDiscriminator()

    return generator, discriminator


def medmnist_dataloader_factory(
    data_flag: str = "bloodmnist",
    batch_size: int = 32,
) -> Tuple[data.DataLoader, data.DataLoader, data.DataLoader]:
    info = INFO[data_flag]
    task = info['task']
    n_channels = info['n_channels']
    n_classes = len(info['label'])

    DataClass = getattr(medmnist, info['python_class'])

    data_transform = transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.ToTensor()
    ])

    train_dataset = DataClass(split='train', transform=data_transform, download=True)
    val_dataset = DataClass(split='val', transform=data_transform, download=True)
    test_dataset = DataClass(split='test', transform=data_transform, download=True)

    train_loader = data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader

In [ ]:
train_loader, val_loader, test_loader = medmnist_dataloader_factory()

100%|██████████| 35.5M/35.5M [00:03<00:00, 11.6MB/s]


In [ ]:
generator, discriminator = medmnist_gan_model_factory()

In [ ]:
gen_optimizer = optim.Adam(generator.parameters(), lr=1e-4)
discriminator_optimizer = optim.Adam(discriminator.parameters(), lr=1e-5)

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
from scipy.linalg import sqrtm
import numpy as np


class FIDCalculator:
    """
    Class for caclulating Fréchet Inception Distance (FID).
    FID measures the distance between feature distributions of real and generated images.
    Lower is better.
    """

    def __init__(self, device: str ="cuda"):
        self.device = device
        # pre-trained InceptionV3 model for feature extraction
        self.inception = models.inception_v3(pretrained=True, transform_input=False)
        self.inception.fc = nn.Identity()
        self.inception.to(device)
        self.inception.eval()

    def _get_features(self, images: torch.Tensor) -> np.ndarray:
        """Extract features from images using InceptionV3."""
        with torch.no_grad():
            if images.size(-1) != 299:
                images = nn.functional.interpolate(images, size=(299, 299), mode='bilinear', align_corners=False)

            if images.size(1) == 1:
                images = images.repeat(1, 3, 1, 1)

            features = self.inception(images)
            return features.cpu().numpy()

    def calculate_fid(self, real_images: torch.Tensor, fake_images: torch.Tensor) -> float:
        """Calculate FID between real and fake images."""
        real_features = self._get_features(real_images)
        fake_features = self._get_features(fake_images)

        real_mean = np.mean(real_features, axis=0)
        fake_mean = np.mean(fake_features, axis=0)

        real_cov = np.cov(real_features, rowvar=False)
        fake_cov = np.cov(fake_features, rowvar=False)

        mean_diff = real_mean - fake_mean
        mean_norm = np.sum(mean_diff ** 2)

        cov_sqrt = sqrtm(real_cov.dot(fake_cov))
        if np.iscomplexobj(cov_sqrt):
            cov_sqrt = cov_sqrt.real

        fid = mean_norm + np.trace(real_cov + fake_cov - 2 * cov_sqrt)
        return fid


def calculate_fid_batch(
    generator: nn.Module,
    val_loader: data.DataLoader,
    fid_calculator: FIDCalculator,
    latent_dim: int,
    device: str,
) -> float:
    generator.eval()

    real_features_list = []
    fake_features_list = []
    batches_processed = 0

    with torch.no_grad():
        for real_images, _ in val_loader:

            real_images = real_images.to(device)
            batch_size = real_images.size(0)

            z = torch.randn(batch_size, latent_dim, device=device)
            fake_images = generator(z)

            real_features = fid_calculator._get_features(real_images)
            fake_features = fid_calculator._get_features(fake_images)

            real_features_list.append(real_features)
            fake_features_list.append(fake_features)

            batches_processed += 1

    all_real_features = np.concatenate(real_features_list, axis=0)
    all_fake_features = np.concatenate(fake_features_list, axis=0)

    real_mean = np.mean(all_real_features, axis=0)
    fake_mean = np.mean(all_fake_features, axis=0)

    real_cov = np.cov(all_real_features, rowvar=False)
    fake_cov = np.cov(all_fake_features, rowvar=False)

    mean_diff = real_mean - fake_mean
    mean_norm = np.sum(mean_diff ** 2)

    from scipy.linalg import sqrtm
    cov_sqrt = sqrtm(real_cov.dot(fake_cov))
    if np.iscomplexobj(cov_sqrt):
        cov_sqrt = cov_sqrt.real

    fid = mean_norm + np.trace(real_cov + fake_cov - 2 * cov_sqrt)

    return float(fid)


def medmnist_train_eval_loop(
    *,
    generator: nn.Module,
    discriminator: nn.Module,
    gen_optimizer: optim.Optimizer,
    disc_optimizer: optim.Optimizer,
    train_loader: data.DataLoader,
    val_loader: data.DataLoader,
    trial: optuna.Trial | None = None,
    adversarial_loss: nn.Module = nn.MSELoss(),
    epochs: int = 20,
    latent_dim: int = 100,
    lambda_l1: float = 100.0,
    fid_eval_freq: int = 3,
    device: str = "cuda",
):
    generator = generator.to(device)
    discriminator = discriminator.to(device)
    l1_loss = nn.L1Loss()
    fid_calculator = FIDCalculator(device)

    for epoch in range(epochs):
        generator.train()
        discriminator.train()
        train_g_loss = 0.0
        train_d_loss = 0.0

        for real_images, _ in train_loader:
            real_images = real_images.to(device)
            batch_size = real_images.size(0)

            # === discriminator training ===
            disc_optimizer.zero_grad()

            real_labels = torch.full((batch_size, 1), 0.9, device=device)
            real_preds = discriminator(real_images)
            real_loss = adversarial_loss(real_preds, real_labels)

            z = torch.randn(batch_size, latent_dim, device=device)
            fake_images = generator(z).detach()
            fake_labels = torch.full((batch_size, 1), 0.1, device=device)
            fake_preds = discriminator(fake_images)
            fake_loss = adversarial_loss(fake_preds, fake_labels)


            d_loss = (real_loss + fake_loss) / 2
            d_loss.backward()
            disc_optimizer.step()

            # === generator training ===
            gen_optimizer.zero_grad()

            z = torch.randn(batch_size, latent_dim, device=device)
            generated_images = generator(z)
            preds = discriminator(generated_images)

            g_loss_adv = adversarial_loss(preds, real_labels)
            g_loss_l1 = l1_loss(generated_images, real_images)
            g_loss = g_loss_adv + lambda_l1 * g_loss_l1
            g_loss.backward()
            gen_optimizer.step()

            train_g_loss += g_loss.item()
            train_d_loss += d_loss.item()


        train_g_loss /= len(train_loader)
        train_d_loss /= len(train_loader)

        generator.eval()
        discriminator.eval()
        val_g_loss = 0.0
        val_d_loss = 0.0

        with torch.no_grad():
            for real_images, _ in val_loader:
                real_images = real_images.to(device)
                batch_size = real_images.size(0)

                real_labels = torch.full((batch_size, 1), 0.9, device=device)
                real_preds = discriminator(real_images)
                real_loss = adversarial_loss(real_preds, real_labels)

                z = torch.randn(batch_size, latent_dim, device=device)
                fake_images = generator(z).detach()
                fake_labels = torch.full((batch_size, 1), 0.1, device=device)
                fake_preds = discriminator(fake_images)
                fake_loss = adversarial_loss(fake_preds, fake_labels)

                d_loss = (real_loss + fake_loss) / 2
                val_d_loss += d_loss.item()

                z = torch.randn(batch_size, latent_dim, device=device)
                generated_images = generator(z)
                preds = discriminator(generated_images)

                g_loss_adv = adversarial_loss(preds, real_labels)
                g_loss_l1 = l1_loss(generated_images, real_images)
                g_loss = g_loss_adv + lambda_l1 * g_loss_l1

                val_g_loss += g_loss.item()

        val_g_loss /= len(val_loader)
        val_d_loss /= len(val_loader)

        if epoch % fid_eval_freq == 0 or epoch == epochs - 1:
            fid_score = calculate_fid_batch(
                generator,
                val_loader,
                fid_calculator,
                latent_dim,
                device,
            )
            final_fid = -fid_score

            print(f"Epoch {epoch+1}/{epochs} — FID: {fid_score:.4f} | Val Gen Loss: {val_g_loss:.4f} | Val Disc Loss: {val_d_loss:.4f}")

            if trial:
                trial.report(final_fid, epoch)
                if trial.should_prune():
                    raise optuna.TrialPruned()
        else:
            print(f"Epoch {epoch+1}/{epochs} — Val Gen Loss: {val_g_loss:.4f} | Val Disc Loss: {val_d_loss:.4f}")

    return final_fid

In [ ]:
final_fid = medmnist_train_eval_loop(
    generator=generator,
    discriminator=discriminator,
    gen_optimizer=gen_optimizer,
    disc_optimizer=discriminator_optimizer,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=30,
)

Epoch 1/30 — FID: 241.6356 | Val Gen Loss: 10.4458 | Val Disc Loss: 0.1761
Epoch 2/30 — Val Gen Loss: 10.4710 | Val Disc Loss: 0.1520
Epoch 3/30 — Val Gen Loss: 10.4587 | Val Disc Loss: 0.1636
Epoch 4/30 — FID: 216.3840 | Val Gen Loss: 10.4796 | Val Disc Loss: 0.1755
Epoch 5/30 — Val Gen Loss: 10.4528 | Val Disc Loss: 0.1633
Epoch 6/30 — Val Gen Loss: 10.4527 | Val Disc Loss: 0.1680
Epoch 7/30 — FID: 239.5591 | Val Gen Loss: 10.4552 | Val Disc Loss: 0.1634
Epoch 8/30 — Val Gen Loss: 10.4407 | Val Disc Loss: 0.1782
Epoch 9/30 — Val Gen Loss: 10.4444 | Val Disc Loss: 0.1735
Epoch 10/30 — FID: 253.4161 | Val Gen Loss: 10.4489 | Val Disc Loss: 0.1735
Epoch 11/30 — Val Gen Loss: 10.4401 | Val Disc Loss: 0.1718
Epoch 12/30 — Val Gen Loss: 10.4418 | Val Disc Loss: 0.1737
Epoch 13/30 — FID: 254.4176 | Val Gen Loss: 10.4426 | Val Disc Loss: 0.1709
Epoch 14/30 — Val Gen Loss: 10.4402 | Val Disc Loss: 0.1813
Epoch 15/30 — Val Gen Loss: 10.4389 | Val Disc Loss: 0.1798
Epoch 16/30 — FID: 265.0033 |